# Choosing the best week with an integer program

A student picks courses, and each lecture, lab or tutorial group offers several sittings. The site must choose one sitting per group so that the week has the fewest clashes first, then the shortest unbroken run of class, then the fewest days with class. Clashes always come first. The last two pull against each other: packing classes into fewer days tends to lengthen the longest run.

The browser solver in JavaScript answers this by search. This notebook shows the same problem written as an integer program in JuMP with HiGHS, in `week_model.jl`. It builds the model piece by piece, solves two real cases and a small one that can be checked by hand, looks at the 60-minute rule and at the two preferences a student can set, and ends by solving all 27 cases of the shared fixture and comparing each answer with the brute-force one stored there.

## Setup

The cell below activates the project in this folder, loads the model code and points at the fixture. Nothing is installed here; the packages come from `Project.toml` and `Manifest.toml` next to this notebook.

In [1]:
import Pkg
Pkg.activate(@__DIR__; io = devnull)
include(joinpath(@__DIR__, "week_model.jl"))

const FIXTURE = joinpath(@__DIR__, "..", "tests", "fixtures", "week_loads.json")
nothing

The helpers below only print. They turn a case into readable lines and compare a solved case with the fixture.

In [2]:
clock(m) = string(lpad(div(m, 60), 2, '0'), ":", lpad(mod(m, 60), 2, '0'))

function weeks_text(e::Event)
    w = e.weeks
    if isempty(w)
        return "all weeks"
    elseif length(w) == 1
        return "week $(first(w))"
    elseif w == collect(first(w):last(w))
        return "weeks $(first(w))-$(last(w))"
    end
    return "weeks " * join(w, ",")
end

describe(e::Event) = string(e.day, " ", clock(e.start_mins), "-", clock(e.end_mins),
                            " (", weeks_text(e), ")")

# Fixed events, then each group with its sittings (at most `limit` per group).
function show_case(case::Case; limit = 6)
    println("Case ", case.name)
    println("Days to keep free: ", isempty(case.free_days) ? "none" : join(case.free_days, ", "),
            "; a break is ", case.break_mins, " minutes")
    println("Fixed events: ", length(case.fixed))
    for f in case.fixed
        println("  ", f.label, ": ", describe(f))
    end
    println("Groups: ", length(case.groups))
    for g in case.groups
        println("  ", g.group_id, " (", length(g.sittings), " sittings)")
        pairs = collect(zip(g.session_ids, g.sittings))
        for (id, s) in first(pairs, limit)
            println("    session ", id, ": ", describe(s))
        end
        if length(pairs) > limit
            println("    ... and ", length(pairs) - limit, " more")
        end
    end
end

matches(case::Case, got) = got.clashes == case.expected_clashes &&
                           got.blocked == case.expected_blocked &&
                           got.front == case.expected_front

# Solves a case, prints C*, B*, the front and one chosen week per front entry,
# and sets the fixture's answer beside it.
function solve_and_report(case::Case)
    sz = model_size(case)
    println("Model size: ", sz.variables, " variables, ", sz.constraints, " constraints")
    got = solve_case(case)
    println()
    println("C* = ", got.clashes, "   fixture: ", case.expected_clashes)
    println("B* = ", got.blocked, "   fixture: ", case.expected_blocked)
    println("Front (longest run in minutes, days): ", join(string.(got.front), ", "))
    println("Fixture front:                        ", join(string.(case.expected_front), ", "))
    println(matches(case, got) ? "Result: match" : "Result: MISMATCH")
    for (entry, choice) in zip(got.front, got.choices)
        println()
        println("A week with (R, D) = ", entry)
        for g in case.groups
            i = findfirst(==(choice[g.group_id]), g.session_ids)
            println("  ", g.group_id, ": session ", g.session_ids[i], ", ", describe(g.sittings[i]))
        end
    end
    return got
end

solve_and_report (generic function with 1 method)

## The formulation

**The instance.** Fixed events $F$ are classes that cannot move: pinned lectures, or extra sittings the student added. Loose groups $G$ are the lectures, labs and tutorials still to place. Group $g \in G$ has candidate sittings $S_g$, and exactly one is chosen. Every event, fixed or candidate, has a day, a start and end time, and the weeks it runs. An event with no week data is present in every week.

**Weeks, days and slots.** $W$ is the set of weeks that get scored, because the measures are taken in the worst week. Two weeks in which exactly the same events are present score identically for every arrangement, so the code keeps one week per distinct set of present events; this loses nothing and keeps the model small. Days $d$ are the days of the week on which any event falls. Each day is cut into slots $t$ of the largest length that divides every start and end time in the case, so every class covers whole slots. In the shared data that is 60 minutes, or 30 when a half-hour time appears. The slot length also divides the break that ends a run, written $\beta$ below; $\beta$ is 60 minutes unless the student asks for a longer one. `slot` below means that length in minutes.

**Decision variables.** Each is listed with what it means.

- $x_{gs} \in \{0,1\}$ is 1 when sitting $s$ of group $g$ is chosen.
- $c_{ab} \in \{0,1\}$ is 1 when two chosen sittings $a$ and $b$ from different groups clash. There is one for each pair of candidates that collide (same day, overlapping times, a shared week or unknown weeks).
- $o_{wdt} \in [0,1]$ is 1 when something is on in slot $t$ of day $d$ in week $w$.
- $b_{wdt} \in [0,1]$ is 1 when that slot counts towards a run, either because it is occupied or because it sits in a short gap between occupied slots.
- $y_m \in \{0,1\}$ is 1 when some stretch of $m$ consecutive slots is entirely counted towards a run.
- $z_{wd} \in \{0,1\}$ is 1 when anything is on day $d$ in week $w$.
- $C$, $R$ and $D$ are the clashes, the longest run in minutes and the number of days. $C$ and $R$ are expressions in the variables above, and $D$ is an integer variable.

**One sitting per group.**

$$
\sum_{s \in S_g} x_{gs} = 1 \quad \text{for every group } g
$$

**Clashes.** A clash pair is forced to 1 when both of its sittings are chosen:

$$
c_{ab} \ge x_a + x_b - 1
$$

A candidate $a$ that collides with $h_a$ fixed events adds $h_a$ clashes whenever it is chosen. Fixed events that collide with each other add a constant $C_F$ whatever is chosen. So

$$
C = C_F + \sum_{\text{pairs}} c_{ab} + \sum_a h_a \, x_a
$$

**Occupancy.** A slot is occupied if a fixed event covers it, or if a chosen candidate that is present that week covers it:

$$
o_{wdt} \ge 1 \ \text{(fixed event covers the slot)}, \qquad o_{wdt} \ge x_a \ \text{(candidate } a \text{ covers it)}
$$

**Bridging gaps shorter than the break.** A run carries on across a free stretch shorter than the break $\beta$, which is 60 minutes unless a preference says otherwise. A slot counts towards a run if it is occupied, or if it lies between two occupied slots $i < t < j$ with fewer than $\beta$ minutes between them:

$$
b_{wdt} \ge o_{wdt}, \qquad b_{wdt} \ge o_{wdi} + o_{wdj} - 1 \quad \text{when } (j - i - 1)\cdot\text{slot} < \beta
$$

Whatever is on between $i$ and $j$, every free stretch there is shorter still, so the whole of $i$ to $j$ is one run.

**Longest run through the $y$ variables.** For every window of $m$ consecutive slots on one day of one week:

$$
y_m \ge \sum_{t \in \text{window}} b_{wdt} - (m - 1)
$$

The right side is 1 only when every slot in the window counts. A run of $n$ slots contains a full window of every length from 1 to $n$, so $y_1, \dots, y_n$ are all forced to 1. Adding them up gives at least the longest run in slots:

$$
R = \text{slot} \cdot \sum_m y_m
$$

**Days.** A day is on when a fixed event is on it, or a chosen candidate is. $D$ is at least the day count of every week, so at least that of the busiest:

$$
z_{wd} \ge 1 \ \text{(fixed event)}, \qquad z_{wd} \ge x_a \ \text{(candidate } a \text{ on day } d \text{ in week } w\text{)}, \qquad D \ge \sum_d z_{wd} \ \text{ for every week } w
$$

**Why these only need to be forced upward.** Every constraint above puts $c$, $o$, $b$, $y$, $z$ or $D$ on the large side of an inequality, so the constraints set a floor and never a ceiling. That is enough because each of these quantities only ever feeds a minimised objective or a cap. A solver that wants $C$, $R$ or $D$ small has no reason to lift any of them above its floor, and once an objective is held at its minimum a solution cannot hide a larger value in them either. Equalities would need the "if and only if" logic of big-M constraints. The inequalities are simpler and give the same answers.

**The order of the objectives.** The solve in `solve_case` has four steps. Until the section on preferences, no day is kept free, so step 2 changes nothing.

1. Minimise $C$ and call the minimum $C^*$. Then add the constraint $C \le C^*$ for the rest of the solve.
2. Minimise $B$, the number of chosen classes on days the student wants free, and call the minimum $B^*$. Then add $B \le B^*$.
3. For each day limit $k = 1, \dots, 7$, add $D \le k$ and minimise $R$ to get a run $r_k$. Hold $R \le r_k$ and minimise $D$ to get the fewest days $d_k$ that run allows. This gives a pair $(r_k, d_k)$.
4. Drop any pair that a different pair matches or beats on both measures, and any repeat. What is left is the front, sorted by run. Its first entry is the default week: the shortest longest run.

Tracing the front by day limits works because tightening the limit can only lengthen the best run. Each limit finds the best run available with at most that many days, so the front is covered without trying every arrangement.

## Case one: a single best week

`comp-year-two-three` is a real load with eight groups and no fixed events. Here is what is on offer.

In [3]:
case1 = load_case(FIXTURE, "comp-year-two-three")
show_case(case1)

Case comp-year-two-three
Days to keep free: none; a break is 60 minutes
Fixed events: 0
Groups: 8
  COMP 2601|Lecture|all (2 sittings)
    session 19190: Wednesday 14:00-15:00 (weeks 1-12)
    session 19191: Thursday 10:00-11:00 (weeks 1,2,3,4,5,6,8,9,10,11,12)
  COMP 2601|Tutorial|all (3 sittings)
    session 19193: Friday 12:00-13:00 (weeks 1-12)
    session 19194: Friday 14:00-15:00 (weeks 1-12)
    session 19195: Friday 15:00-16:00 (weeks 1-12)
  COMP 2611|Lab|all (5 sittings)
    session 19214: Monday 12:00-14:00 (weeks 1-12)
    session 19215: Monday 17:00-19:00 (weeks 1-12)
    session 19217: Tuesday 17:00-19:00 (weeks 1-12)
    session 19218: Wednesday 09:00-11:00 (weeks 1-12)
    session 19219: Wednesday 11:00-13:00 (weeks 1-12)
  COMP 2611|Lecture|all (2 sittings)
    session 19216: Tuesday 09:00-10:00 (weeks 1-12)
    session 19220: Wednesday 13:00-14:00 (weeks 1-12)
  COMP 3607|Lab|all (1 sittings)
    session 19236: Friday 12:00-14:00 (weeks 1-12)
  COMP 3607|Lecture|all (

The model size, then the solve. The fixture's brute-force answer is printed beside the model's.

In [4]:
got1 = solve_and_report(case1);

Model size: 119 variables, 226 constraints



C* = 0   fixture: 0
B* = 0   fixture: 0
Front (longest run in minutes, days): (120, 3)
Fixture front:                        (120, 3)
Result: match

A week with (R, D) = (120, 3)
  COMP 2601|Lecture|all: session 19190, Wednesday 14:00-15:00 (weeks 1-12)
  COMP 2601|Tutorial|all: session 19195, Friday 15:00-16:00 (weeks 1-12)
  COMP 2611|Lab|all: session 19217, Tuesday 17:00-19:00 (weeks 1-12)
  COMP 2611|Lecture|all: session 19216, Tuesday 09:00-10:00 (weeks 1-12)
  COMP 3607|Lab|all: session 19236, Friday 12:00-14:00 (weeks 1-12)
  COMP 3607|Lecture|all: session 19235, Wednesday 09:00-10:00 (weeks 1-12)
  COMP 3613|Lecture|all: session 19238, Wednesday 11:00-13:00 (weeks 1-12)
  COMP 3613|Tutorial|all: session 19237, Tuesday 15:00-16:00 (weeks 1-12)


Only one pair, (120, 3), is on the front. Every other week with no clashes is matched or beaten by it on both measures, so there is nothing to trade.

## Case two: a trade-off between run and days

### A case small enough to check by hand

`paper-two-by-two` has two groups of two sittings and nothing fixed. All four sittings run in week 1.

In [5]:
paper = load_case(FIXTURE, "paper-two-by-two")
show_case(paper)

Case paper-two-by-two
Days to keep free: none; a break is 60 minutes
Fixed events: 0
Groups: 2
  HAND 0001|Lecture|all (2 sittings)
    session 1: Monday 09:00-10:00 (week 1)
    session 2: Tuesday 09:00-10:00 (week 1)
  HAND 0001|Tutorial|all (2 sittings)
    session 3: Monday 09:30-10:30 (week 1)
    session 4: Monday 10:00-11:00 (week 1)


The pairs of sittings are the four arrangements. The code below asks the model's own clash rule about each of them.

In [6]:
lecture, tutorial = paper.groups
for (i, a) in enumerate(lecture.sittings), (j, b) in enumerate(tutorial.sittings)
    println("Sittings ", lecture.session_ids[i], " and ", tutorial.session_ids[j], ": ",
            events_collide(a, b) ? "clash" : "no clash")
end

Sittings 1 and 3: clash
Sittings 1 and 4: no clash
Sittings 2 and 3: no clash
Sittings 2 and 4: no clash


Reading the four arrangements from the data above:

| Lecture | Tutorial | Clashes | What the week looks like | Run | Days |
|---|---|---|---|---|---|
| 1 (Mon 09:00-10:00) | 3 (Mon 09:30-10:30) | 1 | the two overlap | | |
| 1 (Mon 09:00-10:00) | 4 (Mon 10:00-11:00) | 0 | Monday 09:00 to 11:00, back to back, one run | 120 | 1 |
| 2 (Tue 09:00-10:00) | 3 (Mon 09:30-10:30) | 0 | Tuesday 60 minutes, Monday 60 minutes | 60 | 2 |
| 2 (Tue 09:00-10:00) | 4 (Mon 10:00-11:00) | 0 | Tuesday 60 minutes, Monday 60 minutes | 60 | 2 |

The first arrangement clashes, so $C^* = 0$ and it drops out. The other three reach the pairs (120, 1), (60, 2) and (60, 2). Neither of the two distinct pairs beats the other: (60, 2) has the shorter run but takes two days, and (120, 1) takes one day but has the longer run. So the front is (60, 2) and (120, 1), sorted by run. The model should print exactly that.

In [7]:
got_paper = solve_and_report(paper);

Model size: 24 variables, 37 constraints

C* = 0   fixture: 0
B* = 0   fixture: 0
Front (longest run in minutes, days): (60, 2), (120, 1)
Fixture front:                        (60, 2), (120, 1)
Result: match

A week with (R, D) = (60, 2)
  HAND 0001|Lecture|all: session 2, Tuesday 09:00-10:00 (week 1)
  HAND 0001|Tutorial|all: session 4, Monday 10:00-11:00 (week 1)

A week with (R, D) = (120, 1)
  HAND 0001|Lecture|all: session 1, Monday 09:00-10:00 (week 1)
  HAND 0001|Tutorial|all: session 4, Monday 10:00-11:00 (week 1)


### A real load with the same trade-off

`comp-year-two-three-with-foun` is the case above plus two more groups, one of them with 33 sittings. Only the first few sittings of each group are listed.

In [8]:
case2 = load_case(FIXTURE, "comp-year-two-three-with-foun")
show_case(case2; limit = 4)

Case comp-year-two-three-with-foun
Days to keep free: none; a break is 60 minutes
Fixed events: 0
Groups: 10
  COMP 2601|Lecture|all (2 sittings)
    session 19190: Wednesday 14:00-15:00 (weeks 1-12)
    session 19191: Thursday 10:00-11:00 (weeks 1,2,3,4,5,6,8,9,10,11,12)
  COMP 2601|Tutorial|all (3 sittings)
    session 19193: Friday 12:00-13:00 (weeks 1-12)
    session 19194: Friday 14:00-15:00 (weeks 1-12)
    session 19195: Friday 15:00-16:00 (weeks 1-12)
  COMP 2611|Lab|all (5 sittings)
    session 19214: Monday 12:00-14:00 (weeks 1-12)
    session 19215: Monday 17:00-19:00 (weeks 1-12)
    session 19217: Tuesday 17:00-19:00 (weeks 1-12)
    session 19218: Wednesday 09:00-11:00 (weeks 1-12)
    ... and 1 more
  COMP 2611|Lecture|all (2 sittings)
    session 19216: Tuesday 09:00-10:00 (weeks 1-12)
    session 19220: Wednesday 13:00-14:00 (weeks 1-12)
  COMP 3607|Lab|all (1 sittings)
    session 19236: Friday 12:00-14:00 (weeks 1-12)
  COMP 3607|Lecture|all (2 sittings)
    session 

In [9]:
got2 = solve_and_report(case2);

Model size: 330 variables, 737 constraints



C* = 0   fixture: 0
B* = 0   fixture: 0
Front (longest run in minutes, days): (120, 4), (180, 3)
Fixture front:                        (120, 4), (180, 3)
Result: match

A week with (R, D) = (120, 4)
  COMP 2601|Lecture|all: session 19190, Wednesday 14:00-15:00 (weeks 1-12)
  COMP 2601|Tutorial|all: session 19195, Friday 15:00-16:00 (weeks 1-12)
  COMP 2611|Lab|all: session 19217, Tuesday 17:00-19:00 (weeks 1-12)
  COMP 2611|Lecture|all: session 19216, Tuesday 09:00-10:00 (weeks 1-12)
  COMP 3607|Lab|all: session 19236, Friday 12:00-14:00 (weeks 1-12)
  COMP 3607|Lecture|all: session 19235, Wednesday 09:00-10:00 (weeks 1-12)
  COMP 3613|Lecture|all: session 19238, Wednesday 11:00-13:00 (weeks 1-12)
  COMP 3613|Tutorial|all: session 19237, Tuesday 15:00-16:00 (weeks 1-12)
  FOUN 1101|Lecture|all: session 19936, Thursday 17:00-19:00 (week 1)
  FOUN 1101|Tutorial|all: session 19930, Wednesday 15:00-16:00 (weeks 2-12)

A week with (R, D) = (180, 3)
  COMP 2601|Lecture|all: session 19190, 

Adding the extra groups makes a 120-minute run over 3 days unreachable. Two weeks remain on the front: a 120-minute run across four days, and a 180-minute run across three. Neither beats the other, which is the same shape as the paper case on a larger scale. The site shows the shortest run by default and lets the student trade it for fewer days.

## The 60-minute rule

Two classes belong to the same run when the free time between them is less than 60 minutes. Both cases below have one pinned lecture on Tuesday 09:00 to 10:00 and a tutorial group that can sit on Tuesday or on Thursday. Only the gap between the Tuesday sittings differs.

In [10]:
for name in ["gap-thirty-minutes", "gap-sixty-minutes"]
    case = load_case(FIXTURE, name)
    show_case(case)
    println("Slot length: ", build_model(case)[:slot], " minutes")
    solve_and_report(case)
    println("\n", "-"^60, "\n")
end

Case gap-thirty-minutes
Days to keep free: none; a break is 60 minutes
Fixed events: 1
  HAND 0003 Lecture (pinned): Tuesday 09:00-10:00 (weeks 1-12)
Groups: 1
  HAND 0003|Tutorial|all (2 sittings)
    session 21: Tuesday 10:30-11:30 (weeks 1-12)
    session 22: Thursday 10:30-11:30 (weeks 1-12)
Slot length: 30 minutes
Model size: 23 variables, 36 constraints

C* = 0   fixture: 0
B* = 0   fixture: 0
Front (longest run in minutes, days): (60, 2), (150, 1)
Fixture front:                        (60, 2), (150, 1)
Result: match

A week with (R, D) = (60, 2)
  HAND 0003|Tutorial|all: session 22, Thursday 10:30-11:30 (weeks 1-12)

A week with (R, D) = (150, 1)
  HAND 0003|Tutorial|all: session 21, Tuesday 10:30-11:30 (weeks 1-12)

------------------------------------------------------------

Case gap-sixty-minutes
Days to keep free: none; a break is 60 minutes
Fixed events: 1
  HAND 0004 Lecture (pinned): Tuesday 09:00-10:00 (weeks 1-12)
Groups: 1
  HAND 0004|Tutorial|all (2 sittings)
    ses

With a 30-minute gap the Tuesday tutorial runs 10:30 to 11:30, so the day reads 09:00 to 10:00, a half-hour break, then 10:30 to 11:30. The break is shorter than 60 minutes, so the two classes join into one run of 150 minutes on one day. Choosing Thursday instead gives two runs of 60 minutes but costs a second day, so the front keeps both, (60, 2) and (150, 1). With a 60-minute gap the Tuesday tutorial starts at 11:00, the break is exactly 60 minutes and the run ends there. Tuesday is now two separate runs of 60 minutes on a single day, which beats the Thursday option on days and ties it on run, so the front is just (60, 1).

## Two preferences: a longer break and days to keep free

The Re-arrange dialog lets a student change two things, and each case in the fixture may carry them. A case with neither keeps no day free and a break of 60 minutes, which is the model above.

**The break.** The student can count a break as 2 hours instead of 1. The constant 60 becomes the case's own $\beta$. The slot length must divide $\beta$ as well as every time, and the bridging rule reads

$$
b_{wdt} \ge o_{wdi} + o_{wdj} - 1 \quad \text{when } (j - i - 1)\cdot\text{slot} < \beta
$$

so with 2 hours, two classes 90 minutes apart are one run.

**Days to keep free.** A new count $B$ is the number of chosen loose sittings that fall on a free day:

$$
B = \sum_{a \text{ on a free day}} x_a
$$

Fixed events never count, because they cannot move.

**Where it goes in the solve.** $B$ is a new level between clashes and the day-limit sweep: minimise $C$ and hold it, minimise $B$ and hold it, then trace the front over run and days as before. Clashes still come first because a missed class is worse than a class on a day the student wanted free. So a free day can never cause a clash: when the only sitting that avoids a clash is on a free day, it is chosen.

First the break. `gap-ninety-two-hour-break` has a pinned lecture on Thursday 09:00 to 10:30, a pinned lab at 14:30 to 15:30, and a tutorial that can sit at 12:00, at 12:30 or on Friday at 08:00.


In [11]:
case_gap = load_case(FIXTURE, "gap-ninety-two-hour-break")
show_case(case_gap)
println("Slot length: ", build_model(case_gap)[:slot], " minutes")
solve_and_report(case_gap);


Case gap-ninety-two-hour-break
Days to keep free: none; a break is 120 minutes
Fixed events: 2
  HAND 0007 Lecture (pinned): Thursday 09:00-10:30 (weeks 1-12)
  HAND 0007 Lab (pinned): Thursday 14:30-15:30 (weeks 1-12)
Groups: 1
  HAND 0007|Tutorial|all (3 sittings)
    session 71: Thursday 12:00-13:00 (weeks 1-12)
    session 72: Thursday 12:30-13:30 (weeks 1-12)
    session 73: Friday 08:00-09:00 (weeks 1-12)
Slot length: 30 minutes
Model size: 44 variables, 134 constraints



C* = 0   fixture: 0
B* = 0   fixture: 0
Front (longest run in minutes, days): (90, 2), (180, 1)
Fixture front:                        (90, 2), (180, 1)
Result: match

A week with (R, D) = (90, 2)
  HAND 0007|Tutorial|all: session 73, Friday 08:00-09:00 (weeks 1-12)

A week with (R, D) = (180, 1)
  HAND 0007|Tutorial|all: session 72, Thursday 12:30-13:30 (weeks 1-12)


With a 2-hour break the 12:30 tutorial is 120 minutes after the lecture, which is enough to end a run, but only 60 minutes before the lab, so it joins the lab: a run of 180 minutes from 12:30 to 15:30, all on Thursday. The Friday tutorial gives a longest run of 90 minutes, the lecture alone, at the price of a second day. Both are on the front. The 12:00 tutorial is 90 minutes from each pinned class, so it would join all three into one run, and the 12:30 sitting beats it. The slot length is 30 minutes here, since the half-hour times must be whole slots.

The same case with the break put back to 60 minutes, built by copying the case and changing only that field:


In [12]:
case_gap_hour = Case(case_gap.name, case_gap.fixed, case_gap.groups, case_gap.free_days, 60,
                     case_gap.expected_clashes, case_gap.expected_blocked, case_gap.expected_front)
println("Front with a 60-minute break: ", join(string.(solve_case(case_gap_hour).front), ", "))
println("Front with a 2-hour break:    ", join(string.(case_gap.expected_front), ", "))


Front with a 60-minute break: (90, 1)
Front with a 2-hour break:    (90, 2), (180, 1)


With a 60-minute break, gaps of 90 minutes already end a run, so a Thursday tutorial leaves the lecture, the tutorial and the lab as three separate runs and the longest is 90 minutes on one day. That beats the Friday option outright, so one entry is left. The 2-hour break is what makes the choice a real trade.

Now a free day. `paper-clash-free-on-free-day` has a pinned lecture on Tuesday 09:00 to 11:00, a lab group with a sitting on Monday and one on Tuesday, and a tutorial group with a sitting on Monday and one on Wednesday. Monday is a free day.


In [13]:
case_free = load_case(FIXTURE, "paper-clash-free-on-free-day")
show_case(case_free)
solve_and_report(case_free);


Case paper-clash-free-on-free-day
Days to keep free: Monday; a break is 60 minutes
Fixed events: 1
  HAND 0006 Lecture (pinned): Tuesday 09:00-11:00 (weeks 1-12)
Groups: 2
  HAND 0006|Lab|all (2 sittings)
    session 61: Monday 10:00-11:00 (weeks 1-12)
    session 62: Tuesday 10:00-11:00 (weeks 1-12)
  HAND 0006|Tutorial|all (2 sittings)
    session 63: Monday 14:00-15:00 (weeks 1-12)
    session 64: Wednesday 14:00-15:00 (weeks 1-12)
Model size: 20 variables, 25 constraints

C* = 0   fixture: 0
B* = 1   fixture: 1
Front (longest run in minutes, days): (120, 3)
Fixture front:                        (120, 3)
Result: match

A week with (R, D) = (120, 3)
  HAND 0006|Lab|all: session 61, Monday 10:00-11:00 (weeks 1-12)
  HAND 0006|Tutorial|all: session 64, Wednesday 14:00-15:00 (weeks 1-12)


The Tuesday lab sits inside the pinned lecture, so it would add a clash. The Monday lab avoids it but lands on the free day. The model picks the Monday lab, so $C^* = 0$ and $B^* = 1$: clashes are settled first, and only then is $B$ minimised. For the tutorial there is no such pressure, so it takes Wednesday and keeps $B$ at 1. Had free days been ranked above clashes, the lab would have gone to Tuesday and the week would carry a clash to save a Monday class.


## How the browser solver gets the same answers

The browser solver in `assets/js/week-optimiser.js` does not use an integer program. It searches depth first, choosing one sitting per group in turn, and abandons a partial week as soon as a finished week already matches or beats it on clashes, classes on free days, longest run and days. That is safe because all four can only grow as classes are added. To keep each step cheap it scores only one representative week for each distinct set of classes present, as the Julia code does. It stops after a fixed count of steps, `NODE_BUDGET`, rather than after a time limit, so the same courses give the same week on any machine. The two solvers were written separately and agree on every shared case, which is the check that both implement the same definition of the best week.

`check_fixture.jl` runs all 27 cases through the Julia model and exits 0 only if every one matches on $C^*$, $B^*$ and the front. From the project root:

```
julia --project=optimisation optimisation/check_fixture.jl
```

## All 27 cases

The last cell loads every case in the fixture, solves it and compares its clashes, classes on free days and front with the fixture's answers.

In [14]:
function check_all(path)
    cases = load_cases(path)
    matched = 0
    for case in cases
        got = solve_case(case)
        ok = matches(case, got)
        matched += ok ? 1 : 0
        println(rpad(case.name, 50), "C* = ", got.clashes, "  B* = ", got.blocked, "  front ",
                rpad(join(string.(got.front), " "), 32), ok ? "match" : "MISMATCH")
    end
    println()
    println(matched, " of ", length(cases), " cases match the fixture")
end

check_all(FIXTURE)

one-arrangement                                   C* = 0  B* = 0  front (120, 3)                        match
comp-year-two-three                               C* = 0  B* = 0  front (120, 3)                        match
comp-year-two-three-pinned-lecture                C* = 0  B* = 0  front (180, 4) (240, 3)               match


comp-year-two-three-with-foun                     C* = 0  B* = 0  front (120, 4) (180, 3)               match


first-year-large                                  C* = 0  B* = 0  front (120, 3) (360, 2)               match
agbu-weeks-differ                                 C* = 0  B* = 0  front (60, 1)                         match


agbu-with-biol                                    C* = 0  B* = 0  front (60, 2) (120, 1)                match
split-lecture                                     C* = 0  B* = 0  front (120, 4)                        match


unavoidable-clash                                 C* = 1  B* = 0  front (120, 4)                        match
pinned-extra-sitting                              C* = 0  B* = 0  front (120, 3)                        match


biol-alternate-weeks                              C* = 0  B* = 0  front (240, 3)                        match


psychology-and-foundation                         C* = 0  B* = 0  front (120, 2)                        match


sciences-mixed                                    C* = 0  B* = 0  front (120, 2)                        match
sciences-mixed-monday-friday-free                 C* = 0  B* = 0  front (120, 2)                        match
comp-year-two-three-tuesday-free                  C* = 0  B* = 0  front (240, 4)                        match


comp-year-two-three-monday-friday-free            C* = 0  B* = 2  front (120, 3)                        match
comp-year-two-three-pinned-lecture-wednesday-free C* = 0  B* = 1  front (180, 5)                        match
comp-year-two-three-two-hour-break                C* = 0  B* = 0  front (180, 5) (240, 4) (360, 3)      match
agbu-with-biol-both-choices                       C* = 0  B* = 0  front (240, 2) (360, 1)               match
paper-two-by-two                                  C* = 0  B* = 0  front (60, 2) (120, 1)                match
paper-forced-clash                                C* = 1  B* = 0  front (120, 1)                        match


gap-thirty-minutes                                C* = 0  B* = 0  front (60, 2) (150, 1)                match
gap-sixty-minutes                                 C* = 0  B* = 0  front (60, 1)                         match
uploaded-no-weeks                                 C* = 0  B* = 0  front (120, 2)                        match
uploaded-beside-dated                             C* = 0  B* = 0  front (120, 2) (210, 1)               match
paper-clash-free-on-free-day                      C* = 0  B* = 1  front (120, 3)                        match
gap-ninety-two-hour-break                         C* = 0  B* = 0  front (90, 2) (180, 1)                match

27 of 27 cases match the fixture
